In [1]:
from PIL import Image, ImageEnhance
import math
import os

def img_watermark(image_name, image_path):
    # 경로 설정
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    os.makedirs(output_dir, exist_ok=True)

    # 1. 원본 이미지 처리
    original = Image.open(image_path)
    file_ext = os.path.splitext(image_name)[1].lower()
    
    # JPG 대응: RGB 모드로 변환
    if original.mode != 'RGBA':
        image = original.convert('RGBA')
    else:
        image = original.copy()

    # 2. 워터마크 로고 준비 (한 번만 로드)
    logo = Image.open("logo.png").convert("RGBA")
    alpha = logo.split()[3]
    alpha = ImageEnhance.Brightness(alpha).enhance(0.6)
    logo.putalpha(alpha)
    logo_width, logo_height = logo.size

    # 3. 워터마크 배치 계산
    width, height = image.size
    interval_x = math.trunc(width / 35)*10 if width > 600 else 200
    interval_y = 200 if height > 600 else math.trunc(height / 30)*10

    padding = 15
    usable_height = height - 2 * padding - logo_height
    num_lines = max(2, int(usable_height // interval_y) + 1)

    # 4. 워터마크 레이어 생성
    watermark_layer = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    if num_lines == 2:
        y_coords = [padding, height - padding - logo_height]
    else:
        step = usable_height / (num_lines - 1)
        y_coords = [int(padding + i * step) for i in range(num_lines)]

    for y in y_coords:
        for x in range(0, width + interval_x, interval_x):
            watermark_layer.paste(logo, (x, y), logo)

    # 5. 회전 처리 (크기 유지)
    rotated_watermark = watermark_layer.rotate(
        45, 
        expand=False,  # 크기 변경 없음
        center=(width//2, height//2)
    )

    # 6. 이미지 합성
    watermarked = Image.alpha_composite(image, rotated_watermark)

    # 7. 저장 모드 결정
    save_path = os.path.join(output_dir, f"wm_{image_name}")
    
    if file_ext in ('.jpg', '.jpeg'):
        watermarked = watermarked.convert('RGB')  # 알파 채널 제거
        watermarked.save(save_path, quality=95, optimize=True)
    else:
        watermarked.save(save_path)

    return save_path


In [2]:
# !pip install PyMuPDF
import os
import fitz  # PyMuPDF 임포트
def pdf_watermark(pdf_name, path):
    
    # 상위 폴더 경로 계산
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    
    # 출력 폴더 생성 (없을 경우)
    os.makedirs(output_dir, exist_ok=True)
    
    # 파일 경로 설정
    original_file = path
    watermark_file = 'WaterMark.pdf'
    new_file = os.path.join(output_dir, f"wm_{pdf_name}")
    
    # PDF 워터마킹 처리
    original_pdf = fitz.open(original_file)
    watermark_pdf = fitz.open(watermark_file)
    
    for page_num in range(len(original_pdf)):
        page = original_pdf[page_num]
        page.show_pdf_page(page.rect, watermark_pdf, 0)
    
    original_pdf.save(new_file)
    return new_file  # 전체 저장 경로 반환

In [3]:
# !pip install mysql-connector-python
import mysql.connector
from collections import defaultdict
import json
from datetime import datetime

# ───────── DB 설정─────────
DB_CONFIG = {
    "host":     "localhost",
    "user":     "root",
    "password": "",
    "database": "idealink",
    "charset":  "utf8mb4"
}

# ───────── 불용어 사전 ─────────
STOPWORDS = {
    "그리고","하기","있는","으로","를","가","이","에",
    "은","는","다","한","하는","또는","하지만","보다"
}

# ───────── DB 연결 헬퍼 ─────────
def connect_db():
    return mysql.connector.connect(**DB_CONFIG)

# ───────── 조회수 상위 20개 summary + views 가져오기 ─────────
def get_top_20_summary_views():
    query = "SELECT summary, view_count FROM post ORDER BY view_count DESC LIMIT 40"
    with connect_db() as conn, conn.cursor() as cur:
        cur.execute(query)
        return cur.fetchall()                   # [(summary, views), ...]

# ───────── 단어별 조회수 합산 키워드 추출 ─────────
def extract_keywords(summary_views, top_k=40):
    word_score = defaultdict(int)
    
    for summary, views in summary_views:
        if not summary:
            continue
        for word in summary.split():
            if word in STOPWORDS:
                continue
            word_score[word] += views

    keywords = sorted(word_score.items(), 
                     key=lambda x: x[1], 
                     reverse=True)[:top_k]
    
    # 프론트엔드 요구사항에 맞게 딕셔너리 형태로 변환
    return [{"word": k[0], "score": k[1]} for k in keywords]

def refresh_keywords():
    global KEYWORDS_DATA
    rows = get_top_20_summary_views()
    KEYWORDS_DATA = extract_keywords(rows)  # 전역 변수 업데이트
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] ✅ 키워드 데이터 갱신 완료")


In [ ]:
# !pip install flask
# !pip install flask-apscheduler
# !pip install flask-cors
from flask import Flask, request, jsonify
from flask import Response
from flask_cors import CORS
from concurrent.futures import ThreadPoolExecutor
from flask_apscheduler import APScheduler

app = Flask(__name__)
app.config['JSON_AS_ASCII'] = False
CORS(app)
KEYWORDS_DATA = []

# Config 클래스 정의
class Config:
    SCHEDULER_API_ENABLED = True # 스케줄러 API를 활성화함

app.config.from_object(Config()) # Flask 앱에 정의한 Config 적용
scheduler = APScheduler() # APScheduler 인스턴스 생성
scheduler.init_app(app) # 생성한 스케줄러를 Flask 앱에 등록(초기화)

# 1시간마다 키워드 자동 갱신
# @scheduler.task('interval', id='refresh_keywords', hours=1)
# 테스트용 1분마다 키워드 자동 갱신
@scheduler.task('interval', id='refresh_keywords', minutes=1)
def scheduled_refresh():
    refresh_keywords()

scheduler.start()

@app.route('/watermark', methods=['GET'])
def watermark():
    try:
        files = request.get_json()['files']
        print("========================================================")
        print("받은 files:", files)
        if not files or not isinstance(files, list):
            return jsonify({"error": "Invalid file list"}), 400

        file_info = [
            (file['filename'], file['path'], file['path'].split(".")[-1].lower())
            for file in files
        ]

        print("========================================================")
        print("정제한 files:", file_info)

        # 쓰레드를 통해 다중 처리
        processed_paths = []
        with ThreadPoolExecutor() as executor:
            futures = []
            for filename, path, ext in file_info:
                if ext == 'pdf':
                    futures.append(executor.submit(pdf_watermark, filename, path))
                else:
                    futures.append(executor.submit(img_watermark, filename, path))
            
            for future in futures:
                result = future.result()
                processed_paths.append(result)

        return jsonify({"wm_path": processed_paths}), 200

    except Exception as e:
        print("워터마크 오류 : ", e)
        return jsonify({"error": str(e)}), 500

# ───────── REST API ─────────
@app.route("/keywords")
def api_keywords():
    try:
        return jsonify(KEYWORDS_DATA)
    except Exception as e:
        print("키워드 오류 : ", e)
        return jsonify({"error": str(e)}), 500


# 서버 가동
if __name__ == "__main__":
    refresh_keywords() # 서버 시작시 키워드 갱신
    app.run()

[2025-06-22 14:13:14] ✅ 키워드 데이터 갱신 완료
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [22/Jun/2025 14:13:40] "GET /keywords HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2025 14:13:56] "GET /keywords HTTP/1.1" 200 -


[2025-06-22 14:14:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:15:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:16:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:17:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:18:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:19:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:20:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:21:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:22:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:23:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:24:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:25:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:26:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:27:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:28:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:29:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:30:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:31:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:32:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:33:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:34:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:35:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:36:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:37:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:38:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 14:39:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 

127.0.0.1 - - [22/Jun/2025 14:41:20] "GET /keywords HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2025 14:41:20] "GET /keywords HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2025 14:41:28] "GET /keywords HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2025 14:41:42] "GET /watermark HTTP/1.1" 200 -


받은 files: [{'fieldname': 'files', 'originalname': 'cat.1014.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'cat.1014.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'cat.1014-1750570901942-238650229.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1014-1750570901942-238650229.jpg', 'size': 17071}]
정제한 files: [('cat.1014-1750570901942-238650229.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1014-1750570901942-238650229.jpg', 'jpg')]


...

받은 files: [{'fieldname': 'files', 'originalname': 'cat.1007.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'cat.1007.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'cat.1007-1750574652934-27976391.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1007-1750574652934-27976391.jpg', 'size': 22455}]
정제한 files: [('cat.1007-1750574652934-27976391.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1007-1750574652934-27976391.jpg', 'jpg')]
[2025-06-22 15:44:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:45:14] ✅ 키워드 데이터 갱신 완료


127.0.0.1 - - [22/Jun/2025 15:45:26] "GET /watermark HTTP/1.1" 200 -


받은 files: [{'fieldname': 'files', 'originalname': 'cat.19.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'cat.19.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'cat.19-1750574726818-628707415.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.19-1750574726818-628707415.jpg', 'size': 12696}]
정제한 files: [('cat.19-1750574726818-628707415.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.19-1750574726818-628707415.jpg', 'jpg')]
[2025-06-22 15:46:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:47:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:48:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:49:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:50:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:51:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:52:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:53:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:54:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:55:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:56:14] ✅ 키워드 데이터 갱신 완료


127.0.0.1 - - [22/Jun/2025 15:56:53] "GET /keywords HTTP/1.1" 200 -


[2025-06-22 15:57:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:58:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 15:59:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:00:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:01:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:02:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:03:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:04:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:05:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:06:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:07:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:08:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:09:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:10:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:11:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:12:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:13:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:14:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:15:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:16:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:17:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:18:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:19:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:20:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:21:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 16:22:14] ✅ 키워드 데이터 갱신 완료
[2025-06-22 